# init

In [1]:
# scripts/run_workspace.py
from workspace import Workspace
from util import create_recipes

# workspace
workspace = Workspace(config_path=["config/base.j2", "config/layout.j2"])
# core
core = workspace.components["core"]

✅ core connected @ 192.168.254.88
🔵 core simulation api enabled
✅ camera connected @ 218622272001
[Display] socket.io connected
[Display] sending initial snapshot (418 items)
[Display] Running at 60 fps


# parameters

In [2]:
# simulation
simulation = False

# speed factor
speed_factor = 1

# cycle
run_per_rack = 8

# hotel
hotel_levels = 4
hotel_joint_j5 = -40
hotel_joint_j6 = 100

# sbs rack adapter
sbs_adapter_j5 = 38 # 10
sbs_adapter_j0 = -50

# syringe
syringe_padding = 60

# tool rack
tool_rack_joint = [-30, 52.866211, -138.88916, -6.745605, -3.713379, -8.920898, 200]

# tube
tube_list = [f"{r}{c}" for r in "ABCDEF" for c in range(1, 9)]
tube_rack_gravity_offset = 4

# cap
cap_list = [f"{r}{c}" for r in "ABCDEF" for c in range(1, 9)]

# capholder
cap_holder_gravity_offset = -15
cap_holder_tool_tcp_z_offset = 1

# decapper
decapper_tool_tcp_z_offset = -3


# main loop

In [3]:
# recepies
rcp = create_recipes(workspace, core, speed_factor=speed_factor)

# simulation
if not simulation:
    core.simulation(False)


# pick plate gripper
rcp["tool_rack_2"].pick()
tool_rack_middle_joint = core.robot_api.joint()
tool_rack_middle_joint[0: 7] = tool_rack_joint[:]
core.robot_api.jmove(joint=tool_rack_middle_joint, 
                    vel=rcp["tool_rack_2"].jmove_vaj[0]*rcp["tool_rack_2"].speed_factor, 
                    accel=rcp["tool_rack_2"].jmove_vaj[1]*rcp["tool_rack_2"].speed_factor, 
                    jerk=rcp["tool_rack_2"].jmove_vaj[2]*rcp["tool_rack_2"].speed_factor)

# level, tube_index
level = 0
cap_index = 0
while True:
    # pick from level {i} of hotel
    rcp["hotel"].pick_from(level)

    # place the sbs plate in
    sbs_adapter_joint = core.robot_api.joint()
    sbs_adapter_joint[5] = sbs_adapter_j5
    sbs_adapter_joint[0] = sbs_adapter_j0
    core.robot_api.jmove(joint=sbs_adapter_joint, 
                        vel=rcp["sbs_adapter"].jmove_vaj[0]*rcp["sbs_adapter"].speed_factor, 
                        accel=rcp["sbs_adapter"].jmove_vaj[1]*rcp["sbs_adapter"].speed_factor, 
                        jerk=rcp["sbs_adapter"].jmove_vaj[2]*rcp["sbs_adapter"].speed_factor)
    rcp["sbs_adapter"].place_in()

    # change the gripper to suction
    tool_rack_middle_joint = core.robot_api.joint()
    tool_rack_middle_joint[0: 7] = tool_rack_joint[:]
    core.robot_api.jmove(joint=tool_rack_middle_joint, 
                        vel=rcp["tool_rack_2"].jmove_vaj[0]*rcp["tool_rack_2"].speed_factor, 
                        accel=rcp["tool_rack_2"].jmove_vaj[1]*rcp["tool_rack_2"].speed_factor, 
                        jerk=rcp["tool_rack_2"].jmove_vaj[2]*rcp["tool_rack_2"].speed_factor)
    rcp["tool_rack_2"].place()
    rcp["tool_rack_1"].pick()

    # middle point before going to the feeder
    rcp["dispense_arm"].above(anchor="center", padding=syringe_padding)
    
    # pick from feeder and place in cap holder
    for index in range(cap_index, cap_index+run_per_rack):
        # above the feeder
        rcp["feeder"].above(anchor="plate_center")
        # detec and present
        rcp["feeder"].present_cap(rcp["inspector"])  
        # pick from feeder
        rcp["feeder"].pick(approach=False)
        # place in cap holder
        rcp["cap_holder"].place_in(cap_list[index], gravity_offset=cap_holder_gravity_offset)
    
    # middle point before going to the tool rack
    rcp["dispense_arm"].above(anchor="center", padding=syringe_padding)

    # place suction gripper
    rcp["tool_rack_1"].place()
    # change the gripper to tube gripper
    rcp["tool_rack_0"].pick()
    tool_rack_middle_joint = core.robot_api.joint()
    tool_rack_middle_joint[0: 7] = tool_rack_joint[:]
    core.robot_api.jmove(joint=tool_rack_middle_joint, 
                        vel=rcp["tool_rack_2"].jmove_vaj[0]*rcp["tool_rack_2"].speed_factor, 
                        accel=rcp["tool_rack_2"].jmove_vaj[1]*rcp["tool_rack_2"].speed_factor, 
                        jerk=rcp["tool_rack_2"].jmove_vaj[2]*rcp["tool_rack_2"].speed_factor)


    # capping
    for index in range(cap_index, cap_index+run_per_rack):
        # pick tube
        rcp["sbs_plate"].pick_from(tube_list[index])
        # place in decapper
        rcp["decapper"].place()   
        # middle point to avoid collision
        rcp["dispense_arm"].above(anchor="center", padding=syringe_padding)
        # pick cap
        rcp["cap_holder"].pick_from(cap_list[index], tool_tcp_z_offset=cap_holder_tool_tcp_z_offset)
        # middle point to avoid collision
        rcp["dispense_arm"].above(anchor="center", padding=syringe_padding)
        # arm down
        rcp["dispense_arm"].down()
        # dispense
        rcp["dispense_arm"].dispense()
        # arm up
        rcp["dispense_arm"].up()
        # capping
        rcp["decapper"].cap(exit=False)
        # pick from decapper
        rcp["decapper"].pick(approach=False, tool_tcp_z_offset=decapper_tool_tcp_z_offset)
        # back to sbs plate
        rcp["sbs_plate"].place_in(tube_list[index], gravity_offset=tube_rack_gravity_offset)  
    
    # place the tube gripper
    tool_rack_middle_joint = core.robot_api.joint()
    tool_rack_middle_joint[0: 7] = tool_rack_joint[:]
    core.robot_api.jmove(joint=tool_rack_middle_joint, 
                        vel=rcp["tool_rack_2"].jmove_vaj[0]*rcp["tool_rack_2"].speed_factor, 
                        accel=rcp["tool_rack_2"].jmove_vaj[1]*rcp["tool_rack_2"].speed_factor, 
                        jerk=rcp["tool_rack_2"].jmove_vaj[2]*rcp["tool_rack_2"].speed_factor)
    rcp["tool_rack_0"].place()
    
    # pick plate gripper
    rcp["tool_rack_2"].pick()
    tool_rack_middle_joint = core.robot_api.joint()
    tool_rack_middle_joint[0: 7] = tool_rack_joint[:]
    core.robot_api.jmove(joint=tool_rack_middle_joint, 
                        vel=rcp["tool_rack_2"].jmove_vaj[0]*rcp["tool_rack_2"].speed_factor, 
                        accel=rcp["tool_rack_2"].jmove_vaj[1]*rcp["tool_rack_2"].speed_factor, 
                        jerk=rcp["tool_rack_2"].jmove_vaj[2]*rcp["tool_rack_2"].speed_factor)

    # pick from sbs adaptor
    sbs_adapter_joint = core.robot_api.joint()
    sbs_adapter_joint[5] = sbs_adapter_j5
    sbs_adapter_joint[0] = sbs_adapter_j0
    core.robot_api.jmove(joint=sbs_adapter_joint, 
                        vel=rcp["sbs_adapter"].jmove_vaj[0]*rcp["sbs_adapter"].speed_factor, 
                        accel=rcp["sbs_adapter"].jmove_vaj[1]*rcp["sbs_adapter"].speed_factor, 
                        jerk=rcp["sbs_adapter"].jmove_vaj[2]*rcp["sbs_adapter"].speed_factor)
    rcp["sbs_adapter"].pick_from(anchor="place")

    # place in level {i} of hotel
    hotel_joint = core.robot_api.joint()
    hotel_joint[5] = hotel_joint_j5
    hotel_joint[6] = hotel_joint_j6
    core.robot_api.jmove(joint=hotel_joint, 
                        vel=rcp["hotel"].jmove_vaj[0]*rcp["hotel"].speed_factor, 
                        accel=rcp["hotel"].jmove_vaj[1]*rcp["hotel"].speed_factor, 
                        jerk=rcp["hotel"].jmove_vaj[2]*rcp["hotel"].speed_factor)
    rcp["hotel"].place_in(level)

    # level
    level = (level+1)%hotel_levels
    
    # update cap_index
    if level == 0:
        cap_index += run_per_rack
    
    # exit condition
    if cap_index >= len(cap_list):
        break

# place the plate gripper
tool_rack_middle_joint = core.robot_api.joint()
tool_rack_middle_joint[0: 7] = tool_rack_joint[:]
core.robot_api.jmove(joint=tool_rack_middle_joint, 
                    vel=rcp["tool_rack_2"].jmove_vaj[0]*rcp["tool_rack_2"].speed_factor, 
                    accel=rcp["tool_rack_2"].jmove_vaj[1]*rcp["tool_rack_2"].speed_factor, 
                    jerk=rcp["tool_rack_2"].jmove_vaj[2]*rcp["tool_rack_2"].speed_factor)
rcp["tool_rack_2"].place()

🟡 core simulation api disabled


KeyboardInterrupt: 